In [ ]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    if "X_train_features.npy" in files:
        print("FOUND:", os.path.join(root, "X_train_features.npy"))

FOUND: /content/drive/MyDrive/processed/processed/X_train_features.npy


In [3]:
from google.colab import drive
drive.mount('/content/drive')
import numpy as np

DATA_PATH = "/content/drive/MyDrive/processed/processed"

X_train_raw = np.load(f"{DATA_PATH}/X_train_features.npy")
train_chan_ids = np.load(f"{DATA_PATH}/train_chan_ids.npy", allow_pickle=True)
X_test = np.load(f"{DATA_PATH}/X_features.npy")
y_test = np.load(f"{DATA_PATH}/y_labels.npy")
test_chan_ids = np.load(f"{DATA_PATH}/chan_ids.npy", allow_pickle=True)

def clean_features(X, chan_ids, y=None, name=""):
    bad_mask = np.isnan(X).any(axis=1) | np.isinf(X).any(axis=1)
    print(f"{name}: {bad_mask.sum()} bad rows out of {len(X)}")
    X_clean = X[~bad_mask]; chan_ids_clean = chan_ids[~bad_mask]
    if y is not None:
        return X_clean, chan_ids_clean, y[~bad_mask]
    return X_clean, chan_ids_clean

X_train_raw, train_chan_ids = clean_features(X_train_raw, train_chan_ids, name="train/")
X_test, test_chan_ids, y_test = clean_features(X_test, test_chan_ids, y_test, name="test/")

print("train/:", X_train_raw.shape, "| test/:", X_test.shape, "| anomalous:", y_test.sum())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
train/: 0 bad rows out of 9599
test/: 0 bad rows out of 25522
train/: (9599, 504) | test/: (25522, 504) | anomalous: 3459


In [4]:
import torch

In [5]:
import random
import torch
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [6]:
import torch, torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, f1_score

SEQ_LEN = 8

class LSTMAutoencoder(nn.Module):
    def __init__(self, d, hidden=32, latent=8):
        super().__init__()
        self.encoder = nn.LSTM(d, hidden, batch_first=True)
        self.to_latent = nn.Linear(hidden, latent)
        self.from_latent = nn.Linear(latent, hidden)
        self.decoder = nn.LSTM(hidden, d, batch_first=True)
    def encode(self, x):
        _, (h, _) = self.encoder(x)
        return self.to_latent(h[-1])
    def forward(self, x):
        z = self.encode(x)
        h_dec = self.from_latent(z).unsqueeze(1).repeat(1, x.size(1), 1)
        out, _ = self.decoder(h_dec)
        return out, z

def make_sequences(feats, seq_len):
    n_seq = len(feats) // seq_len
    return feats[:n_seq*seq_len].reshape(n_seq, seq_len, -1)

def make_seq_labels(labels, seq_len):
    n_seq = len(labels) // seq_len
    lbl = labels[:n_seq*seq_len].reshape(n_seq, seq_len)
    return (lbl.mean(axis=1) > 0.2).astype(int)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [7]:

class ChebConv(nn.Module):
    def __init__(self, in_dim, out_dim, K=3):
        super().__init__()
        self.K = K
        self.weights = nn.ParameterList([nn.Parameter(torch.randn(in_dim, out_dim)*0.1) for _ in range(K)])
    def forward(self, X, L):
        Tx = [X, L @ X]
        for k in range(2, self.K):
            Tx.append(2 * (L @ Tx[-1]) - Tx[-2])
        return sum(Tx[k] @ self.weights[k] for k in range(self.K))

class DeepChebGNN(nn.Module):
    """Two ChebConv layers with a nonlinearity between - more expressive than one linear conv."""
    def __init__(self, in_dim, hidden_dim=16, K=3):
        super().__init__()
        self.conv1 = ChebConv(in_dim, hidden_dim, K)
        self.conv2 = ChebConv(hidden_dim, in_dim, K)
        self.act = nn.ReLU()
    def forward(self, X, L):
        h = self.act(self.conv1(X, L))
        return self.conv2(h, L)
def build_channel_graph(chan_list):
    """Nodes = channels, edges = channels sharing the same letter-prefix subsystem (e.g. E-1, E-2, E-3)."""
    n = len(chan_list)
    adj = np.zeros((n, n))
    for i, ci in enumerate(chan_list):
        for j, cj in enumerate(chan_list):
            if i != j and ci.split("-")[0] == cj.split("-")[0]:
                adj[i, j] = 1.0
    deg = adj.sum(axis=1)
    deg_inv_sqrt = np.diag(1.0 / np.sqrt(np.maximum(deg, 1e-8)))
    L = np.eye(n) - deg_inv_sqrt @ adj @ deg_inv_sqrt - np.eye(n)
    return torch.tensor(L, dtype=torch.float32)

In [11]:
from sklearn.linear_model import LogisticRegression
import random
from sklearn.metrics import accuracy_score, precision_score, recall_score


def make_seq_labels(labels, seq_len):
    n_seq = len(labels) // seq_len
    lbl = labels[:n_seq*seq_len].reshape(n_seq, seq_len)
    return (lbl.mean(axis=1) > 0.2).astype(int)

# ---- Step 1: fix the tune/final split ONCE, so every run is compared fairly ----
channel_splits = {}
for chan in np.unique(test_chan_ids):
    mask_test = test_chan_ids == chan
    yc_test = y_test[mask_test]
    n_seq = len(yc_test) // SEQ_LEN
    if n_seq == 0:
        continue
    seq_labels = make_seq_labels(yc_test, SEQ_LEN)
    n = len(seq_labels)
    perm = np.random.RandomState(123).permutation(n)   # FIXED split seed - same every run
    n_tune = max(1, int(0.4 * n))
    channel_splits[chan] = (perm[:n_tune], perm[n_tune:], seq_labels)

N_RUNS = 5
all_run_tune_probs = []
all_run_final_probs = []
tune_y_fixed, final_y_fixed = None, None

for run in range(N_RUNS):
    print(f"\n========== RUN {run+1}/{N_RUNS} ==========")
    seed = 100 + run
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

    channel_data = {}
    skipped = []

    for chan in np.unique(test_chan_ids):
        mask_train = train_chan_ids == chan
        Xc_train = X_train_raw[mask_train]
        mask_test = test_chan_ids == chan
        Xc_test = X_test[mask_test]
        yc_test = y_test[mask_test]

        if len(Xc_train) < SEQ_LEN*5 or len(Xc_test) < SEQ_LEN or chan not in channel_splits:
            skipped.append(chan)
            continue

        scaler = StandardScaler()
        scaler.fit(Xc_train)
        train_seq = make_sequences(Xc_train, SEQ_LEN)
        test_seq = make_sequences(Xc_test, SEQ_LEN)

        def scale_seq(seq):
            n, s, dd = seq.shape
            return scaler.transform(seq.reshape(-1, dd)).astype("float32").reshape(n, s, dd)
        train_seq_s, test_seq_s = scale_seq(train_seq), scale_seq(test_seq)

        n_val = max(1, int(0.2 * len(train_seq_s)))
        fit_seq, val_seq = train_seq_s[:-n_val], train_seq_s[-n_val:]
        if len(fit_seq) == 0 or len(val_seq) == 0:
            skipped.append(chan)
            continue

        fit_t = torch.tensor(fit_seq).to(device)
        val_t = torch.tensor(val_seq).to(device)
        test_t = torch.tensor(test_seq_s).to(device)

        model = LSTMAutoencoder(Xc_train.shape[1]).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
        criterion = nn.MSELoss()

       # show_epochs = (chan == chan_list_preview) if 'chan_list_preview' in dir() else (list(np.unique(test_chan_ids)).index(chan) == 0)
        show_epochs = (chan == "D-1")
        for epoch in range(80):
            model.train()
            optimizer.zero_grad()
            out, _ = model(fit_t)
            loss = criterion(out, fit_t)
            loss.backward()
            optimizer.step()
            if show_epochs and (epoch+1) % 10 == 0:
                print(f"  [{chan}] Epoch {epoch+1}/80 | Loss: {loss.item():.4f}")

        model.eval()
        with torch.no_grad():
            val_out, val_z = model(val_t)
            val_err = ((val_out - val_t) ** 2).mean(dim=(1, 2)).cpu().numpy()
            test_out, test_z = model(test_t)
            test_err = ((test_out - test_t) ** 2).mean(dim=(1, 2)).cpu().numpy()

        channel_data[chan] = {
            "val_err_mean": val_err.mean(), "val_err_std": val_err.std() + 1e-8,
            "node_feature": val_z.mean(dim=0).cpu().numpy(),
            "test_err": test_err, "test_z_latent": test_z.cpu().numpy(),
        }

    print(f"Trained {len(channel_data)} channels, skipped {len(skipped)}")

    chan_list = list(channel_data.keys())
    L_t = build_channel_graph(chan_list).to(device)
    node_feat_dim = channel_data[chan_list[0]]["node_feature"].shape[0]
    node_features = torch.tensor(np.stack([channel_data[c]["node_feature"] for c in chan_list]), dtype=torch.float32).to(device)

    gnn = DeepChebGNN(node_feat_dim, hidden_dim=16, K=3).to(device)
    opt_g = torch.optim.Adam(gnn.parameters(), lr=1e-2)
    for epoch in range(200):
      gnn.train()
      opt_g.zero_grad()
      refined = gnn(node_features, L_t)
      loss = nn.functional.mse_loss(refined, node_features)
      loss.backward()
      opt_g.step()
      if (epoch + 1) % 10 == 0:
        print(f"  [GNN] Epoch {epoch+1}/200 | Loss: {loss.item():.4f}")


    gnn.eval()
    with torch.no_grad():
        refined_features = gnn(node_features, L_t).cpu().numpy()

    run_tune_X, run_tune_y, run_final_X, run_final_y = [], [], [], []
    for i, chan in enumerate(chan_list):
        cd = channel_data[chan]
        lstm_z = np.clip((cd["test_err"] - cd["val_err_mean"]) / cd["val_err_std"], -10, 10)
        graph_dist = np.linalg.norm(cd["test_z_latent"] - refined_features[i], axis=1)
        graph_z = np.clip((graph_dist - graph_dist.mean())/(graph_dist.std()+1e-8), -10, 10)

        tune_idx, final_idx, seq_labels = channel_splits[chan]
        # guard: sequence count must match what this run actually produced
        if len(lstm_z) != len(seq_labels):
            continue
        run_tune_X.append(np.column_stack([lstm_z[tune_idx], graph_z[tune_idx]]))
        run_tune_y.append(seq_labels[tune_idx])
        run_final_X.append(np.column_stack([lstm_z[final_idx], graph_z[final_idx]]))
        run_final_y.append(seq_labels[final_idx])

    X_tune = np.concatenate(run_tune_X); y_tune_r = np.concatenate(run_tune_y)
    X_final = np.concatenate(run_final_X); y_final_r = np.concatenate(run_final_y)

    combiner = LogisticRegression(class_weight="balanced")
    combiner.fit(X_tune, y_tune_r)
    all_run_tune_probs.append(combiner.predict_proba(X_tune)[:,1])
    all_run_final_probs.append(combiner.predict_proba(X_final)[:,1])
    tune_y_fixed, final_y_fixed = y_tune_r, y_final_r

    run_f1 = f1_score(final_y_fixed, (all_run_final_probs[-1] > np.median(all_run_final_probs[-1])).astype(int), zero_division=0)
    print(f"Run {run+1} single-run F1 (rough): {run_f1:.3f}")

# ---- average across runs, threshold once on the average ----
avg_tune_probs = np.mean(all_run_tune_probs, axis=0)
avg_final_probs = np.mean(all_run_final_probs, axis=0)

candidates = np.percentile(avg_tune_probs, np.arange(50,100,1))
best_thr, best_f1_tune = None, -1
for thr in candidates:
    f1 = f1_score(tune_y_fixed, (avg_tune_probs>thr).astype(int), zero_division=0)
    if f1 > best_f1_tune:
        best_f1_tune, best_thr = f1, thr

final_preds = (avg_final_probs > best_thr).astype(int)
print(f"\n========== ENSEMBLE OF {N_RUNS} RUNS ==========")
print(classification_report(final_y_fixed, final_preds, zero_division=0))
print("Ensemble Accuracy : ", accuracy_score(final_y_fixed, final_preds))


========== RUN 1/5 ==========
  [D-1] Epoch 10/80 | Loss: 0.6490
  [D-1] Epoch 20/80 | Loss: 0.5471
  [D-1] Epoch 30/80 | Loss: 0.5076
  [D-1] Epoch 40/80 | Loss: 0.4563
  [D-1] Epoch 50/80 | Loss: 0.4224
  [D-1] Epoch 60/80 | Loss: 0.3950
  [D-1] Epoch 70/80 | Loss: 0.3793
  [D-1] Epoch 80/80 | Loss: 0.3631
Trained 73 channels, skipped 8
  [GNN] Epoch 10/200 | Loss: 0.0705
  [GNN] Epoch 20/200 | Loss: 0.0277
  [GNN] Epoch 30/200 | Loss: 0.0139
  [GNN] Epoch 40/200 | Loss: 0.0088
  [GNN] Epoch 50/200 | Loss: 0.0060
  [GNN] Epoch 60/200 | Loss: 0.0046
  [GNN] Epoch 70/200 | Loss: 0.0037
  [GNN] Epoch 80/200 | Loss: 0.0031
  [GNN] Epoch 90/200 | Loss: 0.0027
  [GNN] Epoch 100/200 | Loss: 0.0024
  [GNN] Epoch 110/200 | Loss: 0.0022
  [GNN] Epoch 120/200 | Loss: 0.0020
  [GNN] Epoch 130/200 | Loss: 0.0018
  [GNN] Epoch 140/200 | Loss: 0.0017
  [GNN] Epoch 150/200 | Loss: 0.0015
  [GNN] Epoch 160/200 | Loss: 0.0014
  [GNN] Epoch 170/200 | Loss: 0.0013
  [GNN] Epoch 180/200 | Loss: 0.0012
 